# Практичний тренажер: точка на поверхні сфери

## Від дослідження до самостійного розв'язання

Тренажер має **3 режими**:

1. **Дослідження** — студент змінює `R`, `φ`, `θ` і спостерігає результат.
2. **Розрахунок координат** — параметри задані, координати точки приховані. Студент сам обчислює та вводить `X, Y, Z`.
3. **Відновлення проекції** — дві проекції точки задані, третю студент визначає сам.

Основна логіка:

**параметри → просторова точка → координати → три пов'язані проекції**

> NO-CAD: студент не просто керує моделлю, а виконує інженерну дію — обчислює, перевіряє та відновлює геометричну інформацію.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (
    interact, FloatSlider, Checkbox, Dropdown,
    FloatText, Button, HBox, VBox, Output, HTML
)

# ------------------------------------------------------------
# Геометрія сфери
# ------------------------------------------------------------

def point_on_sphere(cx, cy, cz, R, phi, theta):
    ph = np.deg2rad(phi)
    th = np.deg2rad(theta)

    x = cx + R*np.sin(th)*np.cos(ph)
    y = cy + R*np.sin(th)*np.sin(ph)
    z = cz + R*np.cos(th)

    return np.array([x, y, z], dtype=float)

def draw_scene(cx, cy, cz, R, P=None, show_links=True,
               hidden_projection=None, title_extra=""):

    fig = plt.figure(figsize=(15, 8))

    axf = fig.add_subplot(221)
    axp = fig.add_subplot(222)
    axh = fig.add_subplot(223)
    ax3 = fig.add_subplot(224, projection="3d")

    t = np.linspace(0, 2*np.pi, 200)

    # ---------------- FRONT XZ ----------------
    axf.set_title("ФРОНТАЛЬНА ПРОЕКЦІЯ XZ", fontweight="bold")
    axf.set_xlabel("X")
    axf.set_ylabel("Z")
    axf.grid(alpha=.25)
    axf.set_aspect("equal", adjustable="box")
    axf.plot(cx + R*np.cos(t), cz + R*np.sin(t), linewidth=1.8)
    axf.fill(cx + R*np.cos(t), cz + R*np.sin(t), alpha=.07)

    # ---------------- PROFILE YZ ----------------
    axp.set_title("ПРОФІЛЬНА ПРОЕКЦІЯ YZ", fontweight="bold")
    axp.set_xlabel("Y")
    axp.set_ylabel("Z")
    axp.grid(alpha=.25)
    axp.set_aspect("equal", adjustable="box")
    axp.plot(cy + R*np.cos(t), cz + R*np.sin(t), linewidth=1.8)
    axp.fill(cy + R*np.cos(t), cz + R*np.sin(t), alpha=.07)

    # Дзеркальна орієнтація профільної проекції
    axp.invert_xaxis()

    # ---------------- HORIZONTAL XY ----------------
    axh.set_title("ГОРИЗОНТАЛЬНА ПРОЕКЦІЯ XY", fontweight="bold")
    axh.set_xlabel("X")
    axh.set_ylabel("Y")
    axh.grid(alpha=.25)
    axh.set_aspect("equal", adjustable="box")
    axh.plot(cx + R*np.cos(t), cy + R*np.sin(t), linewidth=1.8)
    axh.fill(cx + R*np.cos(t), cy + R*np.sin(t), alpha=.07)

    if P is not None:
        x, y, z = P

        if hidden_projection != "A1":
            axf.scatter(x, z, s=110)
            axf.annotate("A₁", (x,z), xytext=(8,8),
                         textcoords="offset points",
                         fontsize=13, fontweight="bold")

        if hidden_projection != "A2":
            axh.scatter(x, y, s=110)
            axh.annotate("A₂", (x,y), xytext=(8,8),
                         textcoords="offset points",
                         fontsize=13, fontweight="bold")

        if hidden_projection != "A3":
            axp.scatter(y, z, s=110)
            axp.annotate("A₃", (y,z), xytext=(8,8),
                         textcoords="offset points",
                         fontsize=13, fontweight="bold")

        if show_links:
            # Показуємо координатні зв'язки тільки там, де відповідна
            # проекція не прихована.
            if hidden_projection != "A1":
                axf.axvline(x, linestyle="--", linewidth=1, alpha=.3)
                axf.axhline(z, linestyle="--", linewidth=1, alpha=.3)

            if hidden_projection != "A2":
                axh.axvline(x, linestyle="--", linewidth=1, alpha=.3)
                axh.axhline(y, linestyle="--", linewidth=1, alpha=.3)

            if hidden_projection != "A3":
                axp.axvline(y, linestyle="--", linewidth=1, alpha=.3)
                axp.axhline(z, linestyle="--", linewidth=1, alpha=.3)

        # 3D
        u = np.linspace(0, 2*np.pi, 80)
        v = np.linspace(0, np.pi, 45)
        U, V = np.meshgrid(u, v)

        X = cx + R*np.sin(V)*np.cos(U)
        Y = cy + R*np.sin(V)*np.sin(U)
        Z = cz + R*np.cos(V)

        ax3.set_title("3D-МОДЕЛЬ СФЕРИ", fontweight="bold")
        ax3.plot_surface(X, Y, Z, alpha=.18, linewidth=0)

        ax3.plot(cx + R*np.cos(t), cy + R*np.sin(t),
                 np.full_like(t, cz), linestyle="--", linewidth=1)

        ax3.scatter(x, y, z, s=120)
        ax3.text(x, y, z, "  A", fontsize=13, fontweight="bold")

        ax3.plot([cx,x], [cy,y], [cz,z],
                 linestyle=":", linewidth=1.4)

        ax3.scatter(cx, cy, cz, s=35)

        ax3.set_xlabel("X")
        ax3.set_ylabel("Y")
        ax3.set_zlabel("Z")

        span = 2*R
        ax3.set_xlim(cx-span/2, cx+span/2)
        ax3.set_ylim(cy-span/2, cy+span/2)
        ax3.set_zlim(cz-span/2, cz+span/2)

    fig.suptitle(title_extra, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Режим 1. Дослідження
# ------------------------------------------------------------

def research_mode(R, phi, theta, show_links):
    cx, cy, cz = 5, 5, 5
    P = point_on_sphere(cx, cy, cz, R, phi, theta)

    draw_scene(
        cx, cy, cz, R, P, show_links,
        title_extra=(
            f"Дослідження • A({P[0]:.2f}, {P[1]:.2f}, {P[2]:.2f}) • "
            f"R={R:.1f}, φ={phi:.0f}°, θ={theta:.0f}°"
        )
    )

    print(
        f"Координати точки A: "
        f"X={P[0]:.3f}, Y={P[1]:.3f}, Z={P[2]:.3f}"
    )

print("РЕЖИМ 1 — ДОСЛІДЖЕННЯ")
interact(
    research_mode,
    R=FloatSlider(min=2,max=6,step=1,value=4,description="R"),
    phi=FloatSlider(min=0,max=360,step=5,value=45,description="φ"),
    theta=FloatSlider(min=0,max=180,step=5,value=60,description="θ"),
    show_links=Checkbox(value=True,description="Проекційні зв'язки")
)


In [ ]:

# ------------------------------------------------------------
# Режим 2. Розрахунок координат
# ------------------------------------------------------------

print("РЕЖИМ 2 — САМОСТІЙНИЙ РОЗРАХУНОК")

cx, cy, cz = 5, 5, 5
R2 = 4
phi2 = 60
theta2 = 40

target = point_on_sphere(cx, cy, cz, R2, phi2, theta2)

print(f"Дано: центр C({cx},{cy},{cz}), R={R2}, φ={phi2}°, θ={theta2}°")
print("Обчисліть X, Y, Z самостійно та введіть їх нижче.")

x_input = FloatText(value=0, description="X:")
y_input = FloatText(value=0, description="Y:")
z_input = FloatText(value=0, description="Z:")
check_btn = Button(description="Перевірити", button_style="primary")
out = Output()

def check_coordinates(_):
    with out:
        out.clear_output()

        entered = np.array([x_input.value, y_input.value, z_input.value])
        errors = np.abs(entered - target)

        if np.all(errors < 0.05):
            print("✅ Правильно! Координати точки визначені вірно.")
        else:
            print("❌ Є помилка.")
            if errors[0] >= 0.05:
                print(f"Помилка в X. Правильне значення: {target[0]:.3f}")
            if errors[1] >= 0.05:
                print(f"Помилка в Y. Правильне значення: {target[1]:.3f}")
            if errors[2] >= 0.05:
                print(f"Помилка в Z. Правильне значення: {target[2]:.3f}")

        draw_scene(
            cx, cy, cz, R2, target, True,
            title_extra="Правильне положення точки"
        )

check_btn.on_click(check_coordinates)

display(VBox([
    HBox([x_input, y_input, z_input]),
    check_btn,
    out
]))


In [ ]:

# ------------------------------------------------------------
# Режим 3. Відновлення відсутньої проекції
# ------------------------------------------------------------

print("РЕЖИМ 3 — ВІДНОВЛЕННЯ ПРОЕКЦІЇ")

cx, cy, cz = 5, 5, 5
R3 = 4
phi3 = 130
theta3 = 55

target3 = point_on_sphere(cx, cy, cz, R3, phi3, theta3)

projection_choice = Dropdown(
    options=[
        ("A₁ — фронтальна", "A1"),
        ("A₂ — горизонтальна", "A2"),
        ("A₃ — профільна", "A3")
    ],
    description="Приховати:"
)

answer_x = FloatText(value=0, description="коорд. 1:")
answer_y = FloatText(value=0, description="коорд. 2:")
check_proj = Button(description="Перевірити", button_style="primary")
out2 = Output()

def draw_task(*args):
    with out2:
        out2.clear_output()

        hidden = projection_choice.value

        draw_scene(
            cx, cy, cz, R3, target3, True,
            hidden_projection=hidden,
            title_extra="Завдання: відновіть приховану проекцію"
        )

        print("Відсутню проекцію потрібно визначити за двома наявними.")
        print("Введіть її дві координати та натисніть «Перевірити».")

def check_projection(_):
    with out2:
        out2.clear_output()

        hidden = projection_choice.value

        if hidden == "A1":
            target_answer = np.array([target3[0], target3[2]])
        elif hidden == "A2":
            target_answer = np.array([target3[0], target3[1]])
        else:
            # Для профільної проекції графічно вісь Y розвернута,
            # тому вводимо математичні координати Y,Z.
            target_answer = np.array([target3[1], target3[2]])

        entered = np.array([answer_x.value, answer_y.value])
        errors = np.abs(entered - target_answer)

        if np.all(errors < 0.05):
            print("✅ Правильно! Проекцію відновлено.")
        else:
            print("❌ Перевірте координати.")
            print(
                f"Правильна пара координат: "
                f"({target_answer[0]:.3f}, {target_answer[1]:.3f})"
            )

        draw_scene(
            cx, cy, cz, R3, target3, True,
            hidden_projection=None,
            title_extra="Перевірка результату"
        )

check_btn2 = Button(description="Оновити завдання")
check_btn2.on_click(draw_task)
check_proj.on_click(check_projection)

projection_choice.observe(draw_task, names="value")

display(VBox([
    projection_choice,
    HBox([answer_x, answer_y]),
    HBox([check_proj, check_btn2]),
    out2
]))

draw_task()


## Методичний висновок

Тренажер тепер має три рівні діяльності:

**1. Дослідження** → студент спостерігає залежності.

**2. Розрахунок** → студент сам отримує координати просторової точки.

**3. Відновлення** → студент читає систему проекцій та відновлює відсутню інформацію.

Це перетворює інтерактивну модель на **навчальне завдання**, а не просто на демонстрацію.